# Ladder on Kaggle (free T4)

Fine-tunes `Qwen2.5-Coder-3B-Instruct` with QLoRA on Codeforces reasoning traces,
then scores it by running generated programs against real test cases.

**Before you start:** Settings -> Accelerator -> **GPU T4 x2**, and Internet **on**.
Only one T4 is used; Unsloth single-GPU is faster here than splitting a 3B model.

Kaggle gives you 30 GPU-hours/week and kills a session at 12 hours. The config
checkpoints every 100 steps so an interrupted session resumes instead of restarting.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install

Unsloth pins its own compatible torch/transformers/trl set. Let it, rather than
resolving those separately -- a bad resolve here costs 20 minutes of your weekly quota.

In [ ]:
%%capture
!pip install -q -U "unsloth==2025.9.1" "unsloth_zoo==2025.9.1"
!pip install -q "trl>=0.9.6,<0.12" "peft>=0.12.0" "bitsandbytes>=0.43.0"
!git clone -q https://github.com/NiLabs-Org/ladder.git /kaggle/working/ladder
!pip install -q -e /kaggle/working/ladder

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ladder/src")

from ladder.config import load_config

CONFIG = "/kaggle/working/ladder/configs/ladder-3b-t4.yaml"
cfg = load_config(CONFIG)

# Kaggle wipes everything outside /kaggle/working when the session ends.
cfg.train.output_dir = "/kaggle/working/outputs/ladder-3b-t4"
cfg.eval.results_path = "/kaggle/working/outputs/ladder-3b-t4/eval/results.json"

print(cfg.model.base_model, "| ctx", cfg.model.max_seq_length, "| lora r", cfg.model.lora_r)

## 2. Build the SFT data

CPU-bound and streaming, so it does not need the GPU. Roughly 5-10 minutes.
The filter summary it prints is worth reading: if `kept` is a tiny fraction of
`seen`, a bound in the config is too tight.

In [ ]:
from ladder.data.build import build

DATA_DIR = "/kaggle/working/data/sft"
counts = build(cfg, DATA_DIR)
counts

In [ ]:
# Eyeball one example before committing GPU hours to 12,000 of them.
import json

with open(f"{DATA_DIR}/train.jsonl", encoding="utf-8") as fh:
    sample = json.loads(fh.readline())

print("tokens:", sample["n_tokens"])
print(sample["messages"][1]["content"][:900])
print("
--- assistant ---
")
print(sample["messages"][2]["content"][:900])

## 3. Smoke test

20 steps on the 1.5B base. If this completes and the eval below produces verdicts,
the pipeline works and the long run is worth starting. Skip it only if you have
already run it once on this image.

In [ ]:
smoke = load_config("/kaggle/working/ladder/configs/smoke-1.5b.yaml")
smoke.train.output_dir = "/kaggle/working/outputs/smoke"
smoke.eval.results_path = "/kaggle/working/outputs/smoke/eval/results.json"

from ladder.data.build import build as build_smoke
build_smoke(smoke, "/kaggle/working/data/smoke")

from ladder.train.sft import train
train(smoke, "/kaggle/working/data/smoke")

## 4. Train

About 9 hours for one epoch at 8k context on a single T4. Under Kaggle's 12-hour
session cap, but not by much -- if you are close, set `train.max_steps` rather
than letting the session die mid-epoch.

In [ ]:
from ladder.train.sft import train

adapter_dir = train(cfg, DATA_DIR)
adapter_dir

## 5. Evaluate

Two runs on the same held-out problems: the untouched base model, then the
fine-tune. The base number is the only thing that makes the tuned number mean
anything, so do not skip it.

Generated code executes here. Kaggle's VM is disposable, which is why that is
acceptable in this notebook and not on your laptop.

In [ ]:
from ladder.eval.runner import evaluate
from ladder.infer import load_for_inference, make_generator

def score(adapter):
    model, tok = load_for_inference(cfg, adapter)
    label = "tuned" if adapter else "base"
    cfg.eval.results_path = f"/kaggle/working/outputs/eval-{label}.json"
    summary = evaluate(make_generator(model, tok, cfg), cfg)
    del model
    import gc, torch; gc.collect(); torch.cuda.empty_cache()
    return summary

base_summary = score(None)
base_summary["metrics"]

In [ ]:
tuned_summary = score(adapter_dir)
tuned_summary["metrics"]

In [ ]:
print(f"{'model':<12} {'pass@1':>8}")
print(f"{'base':<12} {base_summary['metrics']['pass@1']:>8.3f}")
print(f"{'ladder':<12} {tuned_summary['metrics']['pass@1']:>8.3f}")
print()
print("base verdicts: ", base_summary["verdicts"])
print("tuned verdicts:", tuned_summary["verdicts"])

## 6. Push the adapter

The adapter is ~100MB; the merged model is not pushed, since anyone can merge it
against the Apache-2.0 base themselves.

Add your token as a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

from ladder.infer import load_for_inference
model, tok = load_for_inference(cfg, adapter_dir)
model.push_to_hub("NiLabs-Org/Ladder-3B")
tok.push_to_hub("NiLabs-Org/Ladder-3B")